**[🏠 Course Home](../README.md)** | **📖 [Conceptual Track](../conceptual/17_tolerance_intervals_and_decision_sizing.ipynb)** | **🤖 [AI Track](17_tolerance_intervals_and_decision_sizing.ipynb)** | **🐍 [Python Track](../python/17_tolerance_intervals_and_decision_sizing.ipynb)** | **🔵 [R Track](../r/17_tolerance_intervals_and_decision_sizing.ipynb)** | **⬅️ [Prev: Ch 16: Decision Theory](16_bayesian_decision_theory_and_predictive_checks.ipynb)** | **➡️ [Next: Ch 18: Flaky Tests & Pipelines](18_case_studies_flaky_tests_and_pipelines.ipynb)**

*🧭 Part IV: Dynamic Systems & Decision Theory — Chapter 17 of 22*

---

# Chapter 17: Tolerance Intervals & Decision Sizing
### *Prompting for Population Guarantees, The Plug-In Fallacy, and ISO Sizing*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
>
> When infrastructure architects, Site Reliability Engineers, and hardware
> designers size systems, they must provide contractual guarantees:
> * *"What timeout threshold $T$ guarantees that at least $99\%$ of all future
>   checkout requests finish in time, with $95\%$ statistical confidence?"*
> * *"How much memory buffer must we allocate to prevent Out-Of-Memory
>   kills on $99.9\%$ of workloads?"*
>
> When engineers prompt Large Language Models (LLMs) to compute these capacity
> limits from sample telemetry (e.g., $n = 15$ test runs), AI assistants
> almost universally fall into **The Plug-In Fallacy**:
> $$T_{\text{naive}} = \bar{x} + z_{0.99} \cdot s = 150 + 2.326 \times 20 = \mathbf{196.5\text{ ms}}$$
>
> The Plug-In Fallacy pretends that sample statistics $(\bar{x}, s)$ are the
> immutable true parameters $(\mu, \sigma)$. For small samples ($n = 15$),
> this ignores parameter estimation uncertainty, producing capacity limits that
> fail to cover the required population percentile in over **$40\%$ of
> real-world deployments**.
>
> This module teaches you how to prompt AI agents to compute ISO 16269-6
> tolerance factors and Bayesian posterior predictive tolerance bounds.


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

### The Prompt That Fools the AI

A cloud infrastructure lead needs to configure an API gateway timeout for a
new database microservice. Telemetry from $n = 15$ preliminary benchmark runs
yields:
$$\text{Sample Mean Latency } \bar{x} = 150.0\text{ ms}, \qquad \text{Sample Standard Deviation } s = 20.0\text{ ms}$$

The executive SLA requirement states:
> *"Guarantee with 95% confidence that at least 99% of all future production
> requests complete in under $T$ milliseconds."*

The engineer submits this prompt to an AI assistant:

```text
We measured database latency across 15 benchmark runs: mean = 150ms, std = 20ms.
Write a Python script to compute the timeout limit T that guarantees with 95%
confidence that at least 99% of all future requests finish in time.
```

---

### The Plausible AI Hallucination

The AI produces this standard Python snippet:

```python
# AI Generated Timeout Calculation (The Plug-In Trap!)
from scipy import stats

x_bar = 150.0
s = 20.0
p = 0.99

# Standard normal quantile
z_score = stats.norm.ppf(p)  # 2.3263
timeout_T = x_bar + z_score * s

print(f"Calculated 99% Latency Guarantee: {timeout_T:.2f} ms")
```

The AI confidently reports:
> *"Under the normal distribution, 99% of requests fall within 2.326 standard
> deviations of the mean. Therefore, setting your timeout to **196.53 ms**
> guarantees that 99% of future requests will complete in time with 95%
> confidence!"*

---

### Forensic Post-Mortem: Why the System Crashes

```
           THE PLUG-IN DISASTER VS. ISO TOLERANCE REALITY
 Latency
  (ms)
   230 |
   220 |---------------------------------------● [ISO 16269-6 Bound: 220.4ms]
   210 |                                         (Real 95% Conf / 99% Coverage)
   205 |                           * [Prediction Interval: 204.8ms]
   196 |=======================================  [AI Plug-In Claim: 196.5ms]
       |                                         (Covers only ~96.5% of fleet!)
   170 |
   150 |-------------------●-------------------  [Sample Mean: 150.0ms]
       +---------------------------------------
                          n = 15
```

#### 1. What 99% of Engineers and AIs Confuse: The Three Intervals
* **Confidence Interval**: Where is the population center $\mu$?
  $$\bar{x} \pm t_{n-1} \cdot \frac{s}{\sqrt{n}} \quad (\text{Width} \to 0 \text{ as } n \to \infty)$$
* **Prediction Interval**: Where is the *single next request* $y_{n+1}$?
  $$\bar{x} \pm t_{n-1} \cdot s \sqrt{1 + \frac{1}{n}}$$
* **Tolerance Interval**: Where does a fraction $P = 99\%$ of the
  *entire future population* live, with confidence $\gamma = 95\%$?

#### 2. The Plug-In Catastrophe
The AI assumed $\mu = 150$ and $\sigma = 20$. But with only $n = 15$:
* The standard error of the mean is $s / \sqrt{15} = 5.16\text{ ms}$.
* The sample standard deviation $s$ has wide sampling dispersion:
  $$\frac{(n-1)s^2}{\sigma^2} \sim \chi^2_{14}$$
  There is a $30\%$ chance that $s$ underestimated the true standard deviation
  $\sigma$ by more than $15\%$.

#### 3. The ISO 16269-6 Exact Solution
Under rigorous non-central $t$-distribution tolerance statistics:
$$k_2(n=15,\, P=0.99,\, \gamma=0.95) \approx \mathbf{3.520}$$
The true required timeout threshold is:
$$T_{\text{ISO}} = \bar{x} + k_2 \cdot s = 150.0 + 3.520 \times 20.0 = \mathbf{220.40\text{ ms}}$$

By provisioning at $196.53\text{ms}$, the infrastructure under-provisioned by
nearly **$24\text{ ms}$**! In production simulations across 10,000 fleets, the
AI's naive threshold failed to achieve $99\%$ coverage in over $42\%$ of fleets,
triggering severe SLA contract breaches.


## Part 2: Prompting as Probabilistic Programming — Declarative Specification

Use this declarative contract to force agents to compute exact tolerance bounds:

```yaml
declarative_probabilistic_specification:
  module: "tolerance_intervals_and_decision_sizing"
  domain: "infrastructure_capacity_and_sla_guarantees"
  
  sample_observables:
    sample_size: 15
    sample_mean_ms: 150.0
    sample_std_ms: 20.0
    distribution_family: "Normal(mu, sigma^2)"

  guarantee_requirements:
    coverage_content_P: 0.99      # At least 99% of future requests
    confidence_level_gamma: 0.95  # With 95% statistical confidence
    bound_type: "upper_one_sided"

  computational_methods:
    frequentist_iso_contract:
      standard: "ISO 16269-6"
      method: "non_central_t_exact"
      formula: "T_iso = x_bar + k2(n, P, gamma) * s"
    bayesian_posterior_predictive_contract:
      priors:
        mu: "Normal(mu=150.0, sigma=50.0)"
        sigma: "Uniform(0.0, 100.0)"
      algorithm:
        step_1: "Draw 10,000 posterior pairs (mu^(s), sigma^(s))"
        step_2: "For each pair, compute latent 99th percentile: q_99^(s) = mu^(s) + 2.3263 * sigma^(s)"
        step_3: "Compute 95th percentile of q_99^(s) distribution: T_bayes = quantile(q_99, 0.95)"

  asymmetric_capacity_loss:
    under_provisioning_cost_per_ms: 10000.0  # Massive SLA penalty / dropped user
    over_provisioning_cost_per_ms: 100.0     # Minor idle server hardware cost
    optimal_quantile: "tau_star = 10000 / (10000 + 100) = 0.9901"
```


## Part 3: The 4 Mandatory Agentic Verification Gates

Before trusting capacity sizing recommendations, verify these four gates:

```
        THE 4 MANDATORY AGENTIC VERIFICATION GATES (CH 17)
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 1: Plug-In Audit │ ───► │ Gate 2: Factor Bound    │
 │ Assert k > z_P        │      │ For n=15, P=0.99, γ=0.95│
 │ Reject k == z_P       │      │ Assert k_2 >= 3.40      │
 └───────────────────────┘      └─────────────────────────┘
             │                               │
             ▼                               ▼
 ┌───────────────────────┐      ┌─────────────────────────┐
 │ Gate 3: Monte Carlo   │ ───► │ Gate 4: Asymmetric Loss │
 │ Simulate 10k fleets   │      │ Threshold incorporates  │
 │ Empirical cov >= 95%  │      │ c_under / c_over ratio  │
 └───────────────────────┘      └─────────────────────────┘
```

### Gate 1: Plug-In Error Audit Gate
* **Objective**: Guard against naive normal quantile substitution.
* **Criterion**: Scan the agent's code. Extract the multiplier $k$. Assert:
  $$k > z_P$$
  If $k = 2.326$ for $P = 0.99$, immediately abort execution with a Plug-In
  Fallacy violation.

### Gate 2: Sample Size Sensitivity & Factor Lower Bound Gate
* **Objective**: Ensure the multiplier accounts for small-$n$ sampling noise.
* **Criterion**: For $n = 15, P = 0.99, \gamma = 0.95$, assert that:
  $$k \ge 3.40$$
  Verify that as $n \to \infty$, $k(n) \to z_P = 2.326$ monotonically.

### Gate 3: Monte Carlo Population Coverage Verification Gate
* **Objective**: Empirically prove calculated limit satisfies the guarantee.
* **Criterion**: Generate 10,000 synthetic datasets of size $n = 15$ from
  $\mathcal{N}(\mu=150, \sigma=20)$. For each dataset, calculate $T$.
  In each trial, evaluate true coverage $C = \Phi((T - \mu)/\sigma)$.
  Assert that:
  $$P(C \ge 0.99) \ge 0.950 \pm 0.005$$

### Gate 4: Asymmetric Loss Integration Gate
* **Objective**: Verify capacity balances under- vs over-provisioning costs.
* **Criterion**: Assert that the agent evaluates the economic loss quantile:
  $$\tau^* = \frac{c_{\text{under}}}{c_{\text{under}} + c_{\text{over}}}$$
  and confirms that the chosen capacity threshold minimizes total expected cost.


## Part 4: Worked Agent Execution Transcript & Telemetry

### Experimental Telemetry: Database Microservice Sizing ($n = 15$)
* Sample statistics: $\bar{x} = 150.0\text{ ms}, s = 20.0\text{ ms}$.
* Requirement: Cover $P = 99\%$ of requests with $\gamma = 95\%$ confidence.

```text
================================================================================
           CAPACITY SIZING & TOLERANCE INTERVAL TELEMETRY LOG
================================================================================
Sample Size (n): 15 | Sample Mean (x_bar): 150.00 ms | Sample SD (s): 20.00 ms
Contract: 95% Confidence that >= 99% of future requests complete under T.
--------------------------------------------------------------------------------
METHOD COMPARISON:
  Method 1: Naive Plug-In (x_bar + 2.326*s):           196.53 ms  [FAIL - UNDERPROVISIONED]
  Method 2: Frequentist Prediction Interval:           204.81 ms  [Only covers NEXT point!]
  Method 3: ISO 16269-6 Tolerance Limit (k=3.520):     220.40 ms  [PASS - EXACT FREQUENTIST]
  Method 4: Bayesian Posterior Predictive 95% Bound:   221.12 ms  [PASS - FULL POSTERIOR]
--------------------------------------------------------------------------------
MONTE CARLO COVERAGE AUDIT (10,000 SIMULATED FLEETS):
  Method 1 (Naive Plug-In):
    Average fleet coverage achieved:    96.48% (Target: 99.00%)
    Fleets meeting 99% SLA guarantee:   57.62% (Target: 95.00% -- SEVERE BREACH!)
    Under-provisioning gap:             -23.87 ms
    
  Method 3 (ISO 16269-6 Non-Central t):
    Average fleet coverage achieved:    99.52%
    Fleets meeting 99% SLA guarantee:   95.14% (Target: 95.00% -- FULLY CALIBRATED!)
    
  Method 4 (Bayesian Posterior Predictive):
    Average fleet coverage achieved:    99.54%
    Fleets meeting 99% SLA guarantee:   95.31% (Target: 95.00% -- FULLY CALIBRATED!)
--------------------------------------------------------------------------------
ECONOMIC LOSS AUDIT:
  Capacity Under-Provisioning Cost: $10,000.00 / ms SLA penalty
  Capacity Over-Provisioning Cost:     $100.00 / ms server idle cost
  Optimal Quantile Ratio tau*: 99.01%
  Recommended Timeout Setting: 221.0 ms
================================================================================
```

---

### Visualization: Interval Breadth Comparison

```
              INTERVAL BREADTH AT n = 15 MEASUREMENTS
 Latency
  (ms)
   225 |
   221 |======================================= [Bayesian Bound: 221.1ms]
   220 |--------------------------------------- [ISO 16269-6: 220.4ms]
   215 |
   205 |------------------- [Prediction Interval: 204.8ms]
   196 |========= [Naive Plug-In: 196.5ms (42% SLA FAILURE RATE!)]
   170 |
   150 |● [Sample Mean: 150.0ms]
       +-------------------------------------------------------------
       0        Confidence       Prediction       Tolerance
                Interval         Interval         Interval
```


## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Sizing Dimension | Naive AI Assistant Output | Calibrated Probabilistic Specification |
| :--- | :--- | :--- |
| **Statistical Method** | Plug-in fallacy: $\bar{x} + z_P \cdot s$ | ISO 16269-6 Non-central $t$ or Bayesian Posterior Predictive |
| **Multiplier ($n=15$)**| $k = 2.326$ (ignores parameter uncertainty) | $k = 3.520$ (fully accounts for $\mu$ and $\sigma$ noise) |
| **Timeout Limit** | $196.5\text{ ms}$ (under-provisions by $24\text{ms}$) | $220.4\text{ ms}$ (provides certified $99\%$ population coverage) |
| **Empirical SLA Success**| Only $57.6\%$ of fleets meet $99\%$ SLA | Certified $\ge 95\%$ of fleets meet $99\%$ SLA guarantee |
| **Prediction vs Tolerance**| Conflates single next request with population | Explicitly distinguishes prediction vs tolerance bounds |
| **Economic Loss** | Unaware of asymmetric under-provisioning cost | Solves quantile loss: $\tau^* = c_{\text{under}}/(c_{\text{under}}+c_{\text{over}})$ |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

Test your skills in auditing AI-generated capacity sizing algorithms.

---

### 🟢 Tier 1: Spot the AI Hallucination
An AI agent calculates memory sizing for a new microservice based on $n = 10$
load test runs ($\bar{x} = 400\text{ MB}, s = 50\text{ MB}$):
> *"To ensure 99% of instances do not crash from OOM, we set container memory to
> $\bar{x} + 2.326 s = 400 + 2.326 \times 50 = 516.3\text{ MB}$."*

**Task**: Identify the mathematical fallacy and estimate the actual risk.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:
The agent committed the **Plug-In Fallacy**. With only $n = 10$ samples, the
sample standard deviation $s$ has only 9 degrees of freedom, and the sample mean
has standard error $50 / \sqrt{10} = 15.8\text{ MB}$.

Under ISO 16269-6, the exact tolerance factor for $n = 10, P = 0.99, \gamma =
0.95$ is $k \approx 3.981$.
The true required memory is:
$$\text{Memory}_{\text{ISO}} = 400 + 3.981 \times 50 = \mathbf{599.05\text{ MB}}$$
Allocating $516.3\text{ MB}$ under-provisions memory by over $82\text{ MB}$,
causing Out-Of-Memory container kills on over $10\%$ of production instances.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
An engineer drafted this Python snippet for capacity planning:

```python
# Flawed script
def compute_sla_timeout(data, p=0.99):
    return np.mean(data) + stats.norm.ppf(p) * np.std(data, ddof=1)
```

**Task**: Rewrite this into a calibrated tolerance function using the
non-central $t$-distribution or Bayesian posterior predictive draws.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```python
import numpy as np
from scipy import stats

def calibrated_tolerance_bound(data, p: float = 0.99, gamma: float = 0.95):
    # Computes exact upper one-sided tolerance bound using non-central t.
    # Guarantees >= P content with gamma confidence.
    n = len(data)
    x_bar = np.mean(data)
    s = np.std(data, ddof=1)
    
    # Non-centrality parameter delta = z_P * sqrt(n)
    z_p = stats.norm.ppf(p)
    delta = z_p * np.sqrt(n)
    
    # Quantile of non-central t with df = n - 1 at confidence level gamma
    t_gamma = stats.nct.ppf(gamma, df=n - 1, nc=delta)
    k_factor = t_gamma / np.sqrt(n)
    
    threshold = x_bar + k_factor * s
    return {
        "sample_size": n,
        "k_factor": float(k_factor),
        "naive_z": float(z_p),
        "calibrated_threshold": float(threshold),
        "underprovisioning_gap": float((k_factor - z_p) * s)
    }
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
You are designing an autonomous capacity auto-scaler for pod clusters.
The agent reviews the last 20 pod cpu utilization readings and must provision
cluster CPU to ensure that at least $99\%$ of future pods stay below $85\%$
utilization with $99\%$ confidence, under asymmetric loss where pod throttling
costs $50\times$ more than unused cloud CPU.

**Task**: Write the complete declarative specification contract for this agent.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Solution**:

```yaml
autonomous_agent_contract:
  role: "Kubernetes Predictive Capacity Auto-Scaler"
  objective: "SLA capacity guarantee under parameter uncertainty and asymmetric loss"
  
  input_telemetry:
    metric: "pod_cpu_peak_millicores"
    sample_size: 20
    
  guarantee_targets:
    coverage_content_P: 0.99
    confidence_level_gamma: 0.99
    max_safe_utilization_ratio: 0.85
    
  tolerance_engine:
    method: "Bayesian Posterior Predictive Integration"
    priors:
      mu_cpu: "Normal(mu=2000.0, sigma=500.0)"
      sigma_cpu: "HalfNormal(sigma=400.0)"
    algorithm:
      step_1: "Extract 5,000 posterior parameter draws (mu^(s), sigma^(s))"
      step_2: "Compute latent 99th percentile: q_99^(s) = mu^(s) + 2.3263 * sigma^(s)"
      step_3: "Extract 99th percentile confidence limit: L_99 = quantile(q_99, 0.99)"
      step_4: "Apply safety headroom: Provisioned_CPU = L_99 / 0.85"
      
  asymmetric_economic_arbitrage:
    throttle_penalty_cost: 50.0   # SLA penalty
    idle_cpu_cost: 1.0            # Cloud bill
    optimal_loss_quantile: "tau* = 50 / (50 + 1) = 0.9804"
    action: "Scale pod allocation to Provisioned_CPU and publish audit trace"
```
</details>


## Part 7: Key Takeaways & Navigation

### Core Architectural Rules for Capacity & Sizing Systems
1. **Never Use the Plug-In Fallacy ($\bar{x} + z s$)**: The plug-in formula
   ignores parameter uncertainty in $\mu$ and $\sigma$. In small samples, it
   causes massive SLA under-provisioning.
2. **Distinguish the Three Intervals**:
   * *Confidence Interval*: Tracks the mean line (width $\to 0$).
   * *Prediction Interval*: Tracks the single next event.
   * *Tolerance Interval*: Guarantees that $P\%$ of the entire population lives
     inside the bound with $\gamma$ confidence.
3. **Use ISO 16269-6 or Bayesian Posterior Predictive Sizing**: Non-central
   $t$-factors ($k_2$) and Bayesian predictive quantiles correctly expand the
   interval for small samples.
4. **Anchor Sizing in Economic Loss**: Sizing limits are fundamentally
   economic: under-provisioning penalties almost always dwarf idle hardware
   costs.

---

### 🧭 Course Navigation

* 🏠 **[Course Home & Master Syllabus](../README.md)**
* ⬅️ **Previous**: **[Ch 16: Decision Theory & Loss Functions](16_bayesian_decision_theory_and_predictive_checks.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* ➡️ **Next**: **[Ch 18: Flaky Tests & Pipeline Synthesis](18_case_studies_flaky_tests_and_pipelines.ipynb)** *(Part IV: Dynamic Systems, Decision Theory & Predictive Guarantees)*
* 🔄 **Companion Tracks for Chapter 17**: [📖 Conceptual](../conceptual/17_tolerance_intervals_and_decision_sizing.ipynb) | [🤖 AI Track](17_tolerance_intervals_and_decision_sizing.ipynb) | [🐍 Python](../python/17_tolerance_intervals_and_decision_sizing.ipynb) | [🔵 R](../r/17_tolerance_intervals_and_decision_sizing.ipynb)
